# Protocol 5 — TMS Insular Gating of Ignition Threshold

This notebook simulates the key predictions of
 (Pred 5.A–Pred 5.C):

* **Pred 5.A** — Active insula and thalamic TMS reduce PCI relative to vertex sham
* **Pred 5.B** — Insula TMS selectively abolishes HEP–P3b coupling
* **Pred 5.C** — Thalamic TMS shifts ignition threshold globally across both streams

> All data are synthetic. Parameters come from  in the protocol file.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from scipy.stats import pearsonr

from apgi.core import compute_pi_i_eff, compute_S_t, compute_theta_t, ignition_criterion

## 1 — Protocol 5 parameters

In [ ]:
# From protocol_5_insula_tms.json → apgi_parameters
KAPPA = 100.0
ALPHA = 0.3
BETA = 0.7
DELTA_PI = {
    "vertex_sham": 0.0,  # no TMS effect
    "insula_active": -0.4,  # selectively reduces Πⁱ_eff
    "thalamus_active": -0.3,  # shifts global threshold
}
N_TRIALS = 480
PI_I_BASE = 1.0

## 2 — Simulate three TMS conditions

In [ ]:
def run_condition(delta_pi, seed):
    rng = np.random.default_rng(seed)
    pi_i = max(0.01, PI_I_BASE + delta_pi)
    C_metabolic = rng.uniform(0.5, 2.0, N_TRIALS)
    pi_i_eff = compute_pi_i_eff(pi_i, C_metabolic, kappa=KAPPA)
    pi_e = rng.uniform(0.8, 1.5, N_TRIALS)
    z_e = rng.uniform(0.2, 1.0, N_TRIALS)
    z_i = rng.uniform(0.1, 0.8, N_TRIALS)
    V_info = rng.uniform(0.1, 1.0, N_TRIALS)
    S_t = np.array(
        [
            compute_S_t(
                float(pi_e[t]), float(z_e[t]), float(pi_i_eff[t]), float(z_i[t])
            )
            for t in range(N_TRIALS)
        ]
    )
    theta_t = np.array(
        [
            compute_theta_t(float(C_metabolic[t]), float(V_info[t]), ALPHA, BETA)
            for t in range(N_TRIALS)
        ]
    )
    detected = np.array(
        [ignition_criterion(S_t[t], theta_t[t]) for t in range(N_TRIALS)]
    )
    hep = pi_i_eff + rng.normal(0, 0.08, N_TRIALS)
    p3b = S_t * detected + rng.normal(0, 0.05, N_TRIALS)
    pci = float(detected.mean() * 0.8 + rng.uniform(0, 0.05))
    return dict(detected=detected, hep=hep, p3b=p3b, pci=pci)


conditions = {
    lbl: run_condition(dp, seed=i + 1) for i, (lbl, dp) in enumerate(DELTA_PI.items())
}
for lbl, c in conditions.items():
    r, _ = pearsonr(c["hep"], c["p3b"])
    print(
        f"{lbl:<20} ignition={c['detected'].mean():.2f}  PCI={c['pci']:.3f}  HEP-P3b r={r:.3f}"
    )

## 3 — Visualise predictions Pred 5.A–Pred 5.B

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(13, 4))
labels = list(conditions.keys())
colors = ["#2166ac", "#d6604d", "#9966FF"]

# Pred 5.A: PCI by condition
ax = axes[0]
ax.bar(
    labels,
    [conditions[k]["pci"] for k in labels],
    color=colors,
    alpha=0.85,
    edgecolor="white",
    width=0.4,
)
ax.axhline(0.31, ls="--", lw=1, color="k", alpha=0.6, label="PCI threshold")
ax.set_title("Pred 4.a — PCI by TMS site")
ax.set_ylabel("PCI")
ax.set_ylim(0, 0.8)
ax.legend(fontsize=8)
ax.spines[["top", "right"]].set_visible(False)

# Pred 5.B: Ignition rate
ax = axes[1]
ax.bar(
    labels,
    [conditions[k]["detected"].mean() for k in labels],
    color=colors,
    alpha=0.85,
    edgecolor="white",
    width=0.4,
)
ax.set_title("Pred 5.B — Ignition rate")
ax.set_ylabel("Ignition rate")
ax.set_ylim(0, 1)
ax.spines[["top", "right"]].set_visible(False)

# Pred 5.B: HEP-P3b coupling
ax = axes[2]
r_vals = [pearsonr(conditions[k]["hep"], conditions[k]["p3b"])[0] for k in labels]
ax.bar(labels, r_vals, color=colors, alpha=0.85, edgecolor="white", width=0.4)
ax.axhline(0, ls="--", lw=0.8, color="k", alpha=0.4)
ax.set_title("Pred 5.B — HEP–P3b coupling")
ax.set_ylabel("Pearson r")
ax.spines[["top", "right"]].set_visible(False)

fig.suptitle("Protocol 5 — TMS Insular Gating (Pred 5.A–Pred 5.C)", fontsize=12)
plt.tight_layout()
plt.show()

## 4 — APGI parameter interpretation

The key prediction: insula TMS lowers $\Pi^i_{\text{eff}}$ by ,
which raises $\theta_t$ and reduces the ignition rate and PCI. Thalamic TMS shifts
the threshold globally. Vertex sham provides the baseline.

This maps onto APGI as:
85564\Pi^i_{\text{eff, TMS}} = \Pi^i \cdot e^{-C/\kappa} \cdot e^{\Delta\Pi^i_{\text{TMS}}}85564

| TMS Site | $\Delta\Pi^i$ | Predicted effect |
|---|---|---|
| Vertex (sham) | 0.0 | No change |
| Insula active | −0.4 | Selective interoceptive gating disruption |
| Thalamus active | −0.3 | Global threshold shift |